Dataset Description

**Source:**
Pollution dataset for 5 major Indian cities (2015–2024)

**Rows:**
18265 daily observations

**Features:**
 Pollutants: PM2.5, PM10, NO2, CO, etc.,
 City, Date

**Target Variable:**
 AQI

**Goal:**
The goal of this project is to develop a time-series forecasting model that predicts daily Air Quality Index (AQI) values for major Indian cities up to the next 90 days, starting from 31-12-2024, using historical air pollution data.

Imports 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
import math
import warnings
warnings.filterwarnings("ignore")

Data Loading 

In [ ]:
df=pd.read_csv('/kaggle/input/cityday/city_day.csv')
df

In [ ]:
df.head()

In [ ]:
print("\nData Info:")
df.info()

In [ ]:
print("\nStatistical Summary:")
df.describe()

In [ ]:
#globally defining
pollutants = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene']

Data Cleaning and preprocessing

In [ ]:
df['Date'] = pd.to_datetime(df['Datetime'], errors='coerce')
df. drop('Datetime', axis=1, inplace=True)
df = df.dropna(subset=['Date', 'AQI'])
df = df. sort_values(['City', 'Date'])  
df. set_index('Date', inplace=True)

EDA

In [ ]:
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata']
rows, cols = 2, 3
plt.figure(figsize=(18, 10))

for i, city in enumerate(cities, 1):
    city_df = df[df['City'] == city].copy()
    city_df['AQI_7day_MA'] = city_df['AQI'].rolling(window=7).mean()
    monthly_aqi = city_df['AQI'].resample('M').mean()

    plt.subplot(rows, cols, i)

    plt.plot(city_df.index, city_df['AQI'], alpha=0.4, label='Daily AQI')
    plt.plot(city_df.index, city_df['AQI_7day_MA'], label='7-Day MA', linewidth=1.8)
    plt.plot(monthly_aqi.index, monthly_aqi.values, linestyle='--', linewidth=3, color='darkred', label='Monthly Avg')

    plt.title(f'{city} AQI Trend')
    plt.xlabel('Year')
    plt.ylabel('AQI')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
warnings.filterwarnings('ignore')
print("STATIONARITY TESTS")
print(f"{'City':<12} | {'ADF p-value':<12} | {'KPSS p-value':<12} | {'Stationary?'}")

for city in cities:
    series = df[df['City'] == city]['AQI'].dropna()
    if len(series) < 50:
        continue
    
    adf_p = adfuller(series)[1]
    kpss_p = kpss(series, regression='c', nlags='auto')[1]
    
    # Check if both ADF and KPSS indicate stationarity
    is_stationary = "Yes" if (adf_p < 0.05 and kpss_p >= 0.05) else "Partial/No"
    
    print(f"{city:<12} | {adf_p:<12.4f} | {kpss_p:<12.4f} | {is_stationary}")

# Plot ACF and PACF for a sample city to inspect autocorrelation lag depth
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sample_city = 'Delhi'
city_series = df[df['City'] == sample_city]['AQI'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(city_series, lags=30, ax=axes[0])
axes[0].set_title(f'{sample_city} - ACF')
plot_pacf(city_series, lags=30, ax=axes[1])
axes[1].set_title(f'{sample_city} - PACF')
plt.tight_layout()
plt.show()

print("\nConclusion: Data shows weak autocorrelation - minimal lag features needed")

Feature Engineering

In [ ]:
def get_season(city, month):
    """Get season based on city and month."""
    # Mumbai, Bangalore, Kolkata
    if city in ['Mumbai', 'Bangalore', 'Kolkata']:
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]: 
            return 'Summer'
        elif month in [6, 7, 8, 9]:
            return 'Monsoon'
        else:
            return 'Post-Monsoon'
    
    # Delhi
    elif city == 'Delhi': 
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5, 6]:
            return 'Summer'
        elif month in [7, 8, 9]: 
            return 'Monsoon'
        else:
            return 'Post-Monsoon'
    
    # Chennai
    elif city == 'Chennai':
        if month in [1, 2]: 
            return 'Winter'
        elif month in [3, 4, 5, 6]:
            return 'Summer'
        elif month in [7, 8, 9]: 
            return 'SW-Monsoon'
        else: 
            return 'NE-Monsoon'
    
    # Default
    else:
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]: 
            return 'Summer'
        elif month in [6, 7, 8, 9]: 
            return 'Monsoon'
        else:
            return 'Post-Monsoon'

# Temporal features
df['Month'] = df. index.month
df['Year'] = df.index.year
df['DayOfWeek'] = df.index.dayofweek
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Cyclical encoding for month
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Season
df['Season'] = df.apply(lambda x: get_season(x['City'], x['Month']), axis=1)

# MINIMAL lag features (based on weak autocorrelation finding)
df['AQI_lag1'] = df. groupby('City')['AQI'].shift(1)
df['AQI_roll_mean_7'] = df.groupby('City')['AQI'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
)

# Pollutant ratio (PM2.5 and PM10 are key predictors)
if 'PM2.5' in df.columns and 'PM10' in df.columns:
    df['PM_ratio'] = df['PM2.5'] / (df['PM10'] + 1e-8)

# Drop rows with NaN in key features
df = df.dropna(subset=['AQI', 'AQI_lag1'])

print(f"Dataset shape after feature engineering: {df.shape}")

Train Test Split

In [ ]:
# TARGET SELECTION
y = df['AQI']
X = df.drop(columns=['AQI', 'AQI_Bucket'], errors='ignore')

In [ ]:
# Use last 6 months as test set
split_date = df.index.max() - pd.DateOffset(months=6)

# Time-based split
X_train = X[X.index < split_date]
X_test  = X[X.index >= split_date]

y_train = y[y.index < split_date]
y_test  = y[y.index >= split_date]

# Sanity check
print(f"Train: {X_train.index.min()} → {X_train.index.max()}")
print(f"Test : {X_test.index.min()} → {X_test.index.max()}")

# Save feature columns for later
feature_cols = list(X. columns)
print(f"\nNumber of features: {len(feature_cols)}")

Preprocessing Pipelines

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [ ]:
# Categorical features (only these two)
categorical_features = ['City', 'Season']

# All other columns except target + categorical → numeric
numeric_features = [
    col for col in df.columns
    if col not in categorical_features
    and col not in ['AQI', 'AQI_Bucket']  # remove targets
]

In [ ]:
# Numerical pipeline → Only impute missing values
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical pipeline → impute + OneHotEncode
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
RANDOM_STATE=41

In [ ]:
# Train baseline persistence model, tree-based regressors (Random Forest, XGBoost, LightGBM), and a simple average ensemble.
print("Training models...\n")

# Baseline: Persistence (yesterday's AQI)
y_pred_persist = X_test['AQI_lag1'].values

# Random Forest
rf_model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest trained")

# XGBoost
xgb_model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, 
                                random_state=RANDOM_STATE, n_jobs=-1))
])
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost trained")

# LightGBM
try:
    from lightgbm import LGBMRegressor
    lgbm_model = Pipeline([
        ('preprocess', preprocessor),
        ('regressor', LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                     random_state=RANDOM_STATE, verbose=-1, n_jobs=-1))
    ])
    lgbm_model.fit(X_train, y_train)
    y_pred_lgbm = lgbm_model.predict(X_test)
    print("LightGBM trained")
    HAS_LGBM = True
except ImportError:
    print("LightGBM not installed - skipping")
    HAS_LGBM = False

# Ensemble (simple average across available tree models)
if HAS_LGBM: 
    y_pred_ensemble = (y_pred_rf + y_pred_xgb + y_pred_lgbm) / 3
else:
    y_pred_ensemble = (y_pred_rf + y_pred_xgb) / 2
print("Ensemble created")

Hyperparameter tuning

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
!pip install optuna
import optuna
from optuna.samplers import TPESampler

In [ ]:
# Tune XGBoost hyperparameters using Optuna Bayesian optimization (TPE) with 5-fold TimeSeriesSplit cross-validation.
from sklearn.metrics import mean_squared_error

tscv = TimeSeriesSplit(n_splits=5)
print("\nBayesian Optimization with Optuna...")

# Objective function: evaluates a sampled XGBoost hyperparameter set across TimeSeriesSplit folds and returns mean CV RMSE.
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
    }
    
    model = Pipeline([
        ('preprocess', preprocessor),
        ('regressor', XGBRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    
    scores = []
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        scores.append(rmse)
    
    return np.mean(scores)

sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='minimize', sampler=sampler)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\nOptuna Optimization Complete")
print(f"Best Parameters: {study.best_params}")
print(f"Best CV RMSE: {study.best_value:.2f}")

# Train final tuned XGBoost model with best parameters
best_xgb_optuna = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', XGBRegressor(**study.best_params, random_state=RANDOM_STATE, n_jobs=-1))
])
best_xgb_optuna.fit(X_train, y_train)
y_pred_optuna = best_xgb_optuna.predict(X_test)
optuna_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_optuna))
print(f"Test RMSE: {optuna_test_rmse:.2f}")

# Alias tuned model and predictions for downstream evaluation and deployment cells
best_model = best_xgb_optuna
y_pred_best = y_pred_optuna

Model Evaluation

Compare the Persistence baseline, Random Forest, XGBoost, LightGBM, Ensemble, and Optuna-tuned XGBoost across MAE, RMSE, MAPE, R2, and directional accuracy.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Evaluate models across MAE, RMSE, MAPE, R2, and directional accuracy, then plot comparative bar charts.
def evaluate_model(y_true, y_pred, model_name):
    """Compute MAE, RMSE, MAPE, R2, and day-to-day direction accuracy for a model's predictions."""
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    non_zero = y_true != 0
    mape = np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100
    
    if len(y_true) > 1:
        actual_dir = np.diff(y_true) > 0
        pred_dir = np.diff(y_pred) > 0
        dir_acc = np.mean(actual_dir == pred_dir) * 100
    else: 
        dir_acc = np.nan
    
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 
            'R2': r2, 'Direction_Acc': dir_acc}

results = []
results.append(evaluate_model(y_test, y_pred_persist, 'Persistence'))
results.append(evaluate_model(y_test, y_pred_rf, 'RandomForest'))
results.append(evaluate_model(y_test, y_pred_xgb, 'XGBoost'))
if HAS_LGBM: 
    results.append(evaluate_model(y_test, y_pred_lgbm, 'LightGBM'))
results.append(evaluate_model(y_test, y_pred_ensemble, 'Ensemble'))
results.append(evaluate_model(y_test, y_pred_best, 'Tuned_XGBoost'))

results_df = pd.DataFrame(results)
print("MODEL COMPARISON (Multiple Metrics)")
print(results_df.round(2).to_string(index=False))

best_model_name = results_df.loc[results_df['RMSE'].idxmin(), 'Model']
print(f"\nBest Model: {best_model_name}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
colors = ['#e74c3c' if m == best_model_name else '#3498db' for m in results_df['Model']]
bars = ax1.barh(results_df['Model'], results_df['RMSE'], color=colors)
ax1.set_xlabel('RMSE')
ax1.set_title('Model Comparison - RMSE (Lower is Better)', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

ax2 = axes[1]
ax2.barh(results_df['Model'], results_df['Direction_Acc'], color='#27ae60')
ax2.set_xlabel('Direction Accuracy (%)')
ax2.set_title('Model Comparison - Direction Accuracy (Higher is Better)', fontweight='bold')
ax2.axvline(x=50, color='red', linestyle='--', label='Random (50%)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

Error Analysis, Feature Importance & Residual Diagnostics

Inspect temporal prediction error patterns, top contributing features from the pipeline, city-wise accuracy, and residual distribution/normality.

In [ ]:
# Diagnostic function: plot prediction errors over time, absolute errors, monthly MAE breakdown, and error distribution.
def plot_error_over_time_simple(y_true, y_pred, dates, model_name="Model"):
    """Visualize raw error over time, absolute error, monthly MAE, and error histogram."""
    errors = np.array(y_true) - np.array(y_pred)
    abs_errors = np.abs(errors)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # Error over time
    ax1 = axes[0, 0]
    ax1.plot(dates, errors, alpha=0.6, linewidth=0.8)
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
    rolling_error = pd.Series(errors).rolling(window=7).mean()
    ax1.plot(dates, rolling_error, color='darkred', linewidth=2, label='7-day MA')
    ax1.set_title('Prediction Error Over Time', fontweight='bold')
    ax1.set_ylabel('Error')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Absolute error over time
    ax2 = axes[0, 1]
    ax2.plot(dates, abs_errors, alpha=0.6, color='#e74c3c', linewidth=0.8)
    ax2.axhline(y=np.mean(abs_errors), color='green', linestyle='--', 
                label=f'Mean: {np.mean(abs_errors):.1f}')
    ax2.set_title('Absolute Error Over Time', fontweight='bold')
    ax2.set_ylabel('Absolute Error')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Error by Month
    ax3 = axes[1, 0]
    error_df = pd.DataFrame({'Date': dates, 'AbsError': abs_errors})
    error_df['Month'] = pd.to_datetime(error_df['Date']).dt.month
    monthly_error = error_df.groupby('Month')['AbsError'].mean()
    ax3.bar(monthly_error.index, monthly_error.values, color='#3498db')
    ax3.set_title('Mean Absolute Error by Month', fontweight='bold')
    ax3.set_xlabel('Month')
    ax3.set_ylabel('MAE')
    ax3.grid(axis='y', alpha=0.3)
    
    # Error distribution
    ax4 = axes[1, 1]
    ax4.hist(errors, bins=40, alpha=0.7, color='#27ae60', edgecolor='black')
    ax4.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax4.axvline(x=errors.mean(), color='blue', linestyle='-', 
                label=f'Mean: {errors.mean():.1f}')
    ax4.set_title('Error Distribution', fontweight='bold')
    ax4.set_xlabel('Error')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle(f'Error Analysis: {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_error_over_time_simple(y_test.values, y_pred_best, y_test.index, "Tuned XGBoost")

In [ ]:
# Extract and plot the top N feature importances from the fitted preprocessing + regressor pipeline.
def plot_feature_importance(pipeline, top_n=15):
    """Plot horizontal bar chart of the top N most important features from the trained pipeline."""
    regressor = pipeline.named_steps['regressor']
    feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
    importances = regressor.feature_importances_
    
    fi_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=True).tail(top_n)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(fi_df['Feature'], fi_df['Importance'], color='#3498db')
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'Top {top_n} Feature Importance', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for bar, val in zip(bars, fi_df['Importance']):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, 
                f'{val:.3f}', va='center', fontsize=9)
    
    plt.tight_layout()
    return fig

fig = plot_feature_importance(best_model, top_n=15)
plt.show()

print("\nTop 10 Most Important Features:")
regressor = best_model.named_steps['regressor']
feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': regressor.feature_importances_
}).sort_values('Importance', ascending=False).head(10)
print(fi_df.to_string(index=False))

In [ ]:
# Compare actual vs. predicted AQI values via scatter plot, sample time-series overlay, and city-level error summary.
def plot_actual_vs_predicted(y_true, y_pred, model_name="Model"):
    """Plot actual vs. predicted scatter, time-series slice, and prediction error distribution."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Scatter plot (Actual vs Predicted)
    ax1 = axes[0]
    ax1.scatter(y_true, y_pred, alpha=0.5, s=20, c='#3498db')
    
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    ax1.set_xlabel('Actual AQI', fontsize=12)
    ax1.set_ylabel('Predicted AQI', fontsize=12)
    ax1.set_title(f'{model_name}: Actual vs Predicted', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    r2 = r2_score(y_true, y_pred)
    ax1.text(0.05, 0.95, f'R2 = {r2:.3f}', transform=ax1.transAxes, 
             fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Time series comparison (first 200 points for clarity)
    ax2 = axes[1]
    n_points = min(200, len(y_true))
    ax2.plot(range(n_points), y_true[:n_points], label='Actual', alpha=0.8, linewidth=1.5)
    ax2.plot(range(n_points), y_pred[:n_points], label='Predicted', alpha=0.8, linewidth=1.5)
    ax2.set_xlabel('Sample Index', fontsize=12)
    ax2.set_ylabel('AQI', fontsize=12)
    ax2.set_title('Actual vs Predicted Over Time', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Error distribution
    ax3 = axes[2]
    errors = y_true - y_pred
    ax3.hist(errors, bins=50, alpha=0.7, color='#e74c3c', edgecolor='black')
    ax3.axvline(x=0, color='black', linestyle='--', linewidth=2)
    ax3.axvline(x=errors.mean(), color='blue', linestyle='-', linewidth=2, label=f'Mean Error: {errors.mean():.1f}')
    ax3.set_xlabel('Prediction Error (Actual - Predicted)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Error Distribution', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

y_test_arr = np.array(y_test)
y_pred_arr = np.array(y_pred_best)

fig = plot_actual_vs_predicted(y_test_arr, y_pred_arr, "Tuned XGBoost")
plt.show()

print("\nPerformance by City:")
test_df = X_test.copy()
test_df['Actual'] = y_test
test_df['Predicted'] = y_pred_best
test_df['Error'] = test_df['Actual'] - test_df['Predicted']

city_performance = test_df.groupby('City').agg({
    'Actual': 'mean',
    'Predicted': 'mean',
    'Error': ['mean', 'std']
}).round(2)
print(city_performance)

In [ ]:
# Residual diagnostics: check heteroscedasticity, temporal error patterns, normality, and Q-Q fit.
def plot_residual_analysis(y_true, y_pred, model_name="Model"):
    """Plot residuals vs. predicted values, residuals across samples, density histogram, and Q-Q plot."""
    residuals = np.array(y_true) - np.array(y_pred)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Residuals vs Predicted (check for heteroscedasticity)
    ax1 = axes[0, 0]
    ax1.scatter(y_pred, residuals, alpha=0.5, s=20, c='#3498db')
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax1.set_xlabel('Predicted AQI', fontsize=12)
    ax1.set_ylabel('Residual', fontsize=12)
    ax1.set_title('Residuals vs Predicted Values', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    ax1.text(0.02, 0.98, 'Good: Random scatter around 0\nBad: Funnel or pattern', 
             transform=ax1.transAxes, fontsize=9, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    # Residuals over time/index (check for autocorrelation)
    ax2 = axes[0, 1]
    ax2.plot(range(len(residuals)), residuals, alpha=0.7, linewidth=0.8)
    ax2.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax2.axhline(y=residuals.std()*2, color='orange', linestyle=':', alpha=0.7)
    ax2.axhline(y=-residuals.std()*2, color='orange', linestyle=':', alpha=0.7)
    ax2.set_xlabel('Sample Index', fontsize=12)
    ax2.set_ylabel('Residual', fontsize=12)
    ax2.set_title('Residuals Over Time', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Histogram of residuals (check for normality)
    ax3 = axes[1, 0]
    ax3.hist(residuals, bins=50, alpha=0.7, color='#2ecc71', edgecolor='black', density=True)
    
    from scipy import stats
    x = np.linspace(residuals.min(), residuals.max(), 100)
    ax3.plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()), 
             'r-', linewidth=2, label='Normal Distribution')
    
    ax3.axvline(x=0, color='black', linestyle='--', linewidth=2)
    ax3.set_xlabel('Residual', fontsize=12)
    ax3.set_ylabel('Density', fontsize=12)
    ax3.set_title('Residual Distribution', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Q-Q Plot (check for normality)
    ax4 = axes[1, 1]
    stats.probplot(residuals, dist="norm", plot=ax4)
    ax4.set_title('Q-Q Plot (Normality Check)', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle(f'Residual Analysis: {model_name}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig = plot_residual_analysis(y_test_arr, y_pred_arr, "Tuned XGBoost")
plt.show()

residuals = y_test_arr - y_pred_arr
print("\nResidual Statistics:")
print(f"   Mean Residual: {residuals.mean():.2f} (should be ~0)")
print(f"   Std Residual: {residuals.std():.2f}")
print(f"   Min Residual: {residuals.min():.2f}")
print(f"   Max Residual: {residuals.max():.2f}")
print(f"   % within +/-50 AQI: {(np.abs(residuals) <= 50).mean()*100:.1f}%")
print(f"   % within +/-100 AQI: {(np.abs(residuals) <= 100).mean()*100:.1f}%")

Initial Forecasting Prototype

First iteration of recursive multi-horizon forecasting (30, 60, and 90 days) using a standalone function with seasonal pollutant sampling and expanding 95% confidence intervals.

In [ ]:
# Iteration 1 (Prototype): Standalone recursive forecasting function with seasonal pollutant sampling and confidence bounds.
# (Later consolidated into the final AQIForecastSystem class below.)
def forecast_aqi_with_variability(pipeline, df, city, feature_cols, forecast_days=30):
    """
    Generate multi-day recursive AQI forecasts with seasonal pollutant variation and expanding confidence intervals.
    """
    city_df = df[df['City'] == city].sort_index().copy()
    historical_aqi = city_df['AQI'].tolist()
    
    # Historical volatility for day-to-day variation
    hist_daily_change = city_df['AQI'].diff().dropna()
    change_mean = hist_daily_change.mean()
    change_std = hist_daily_change.std()
    
    # Seasonal pollutant statistics (mean, std, IQR)
    seasonal_stats = {}
    for season in city_df['Season'].unique():
        season_data = city_df[city_df['Season'] == season]
        seasonal_stats[season] = {'pollutants': {}}
        for p in pollutants: 
            if p in city_df.columns:
                p_mean = season_data[p].mean()
                p_std = season_data[p].std()
                seasonal_stats[season]['pollutants'][p] = {
                    'mean': p_mean if pd.notna(p_mean) else city_df[p].mean(),
                    'std': p_std if pd.notna(p_std) else city_df[p].std(),
                    'q25': season_data[p].quantile(0.25),
                    'q75': season_data[p].quantile(0.75)
                }
    
    # Normalized day-of-week multiplier
    dow_pattern = city_df.groupby(city_df.index.dayofweek)['AQI'].mean()
    dow_pattern = dow_pattern / dow_pattern.mean()
    
    future_dates = pd.date_range(
        city_df.index.max() + pd.Timedelta(days=1),
        periods=forecast_days
    )
    
    forecasts = []
    
    for i, date in enumerate(future_dates):
        row = city_df[feature_cols].iloc[-1].to_dict()
        
        row['Month'] = date.month
        row['Year'] = date.year
        row['DayOfWeek'] = date.dayofweek
        row['IsWeekend'] = 1 if date.dayofweek in [5, 6] else 0
        
        if 'Month_sin' in feature_cols:
            row['Month_sin'] = np.sin(2 * np.pi * date.month / 12)
        if 'Month_cos' in feature_cols:
            row['Month_cos'] = np.cos(2 * np.pi * date.month / 12)
        
        season = get_season(city, date.month)
        row['Season'] = season
        row['City'] = city
        
        row['AQI_lag1'] = historical_aqi[-1]
        if 'AQI_roll_mean_7' in feature_cols and len(historical_aqi) >= 7:
            row['AQI_roll_mean_7'] = np.mean(historical_aqi[-7:])
        
        if season in seasonal_stats: 
            for p in pollutants: 
                if p in feature_cols and p in seasonal_stats[season]['pollutants']: 
                    stats = seasonal_stats[season]['pollutants'][p]
                    if pd.notna(stats['mean']) and pd.notna(stats['std']) and stats['std'] > 0:
                        sampled = np.random.normal(stats['mean'], stats['std'] * 0.5)
                        if pd.notna(stats['q25']) and pd.notna(stats['q75']):
                            sampled = np.clip(sampled, stats['q25'], stats['q75'])
                        row[p] = max(0, sampled)
                    elif pd.notna(stats['mean']):
                        row[p] = stats['mean']
        
        if 'PM_ratio' in feature_cols: 
            pm25 = row.get('PM2.5', 0)
            pm10 = row.get('PM10', 1)
            row['PM_ratio'] = pm25 / (pm10 + 1e-8)
        
        row_df = pd.DataFrame([row])[feature_cols]
        base_pred = pipeline.predict(row_df)[0]
        
        dow = date.dayofweek
        if dow in dow_pattern.index:
            base_pred *= dow_pattern[dow]
        
        daily_noise = np.random.normal(change_mean, change_std * 0.5)
        pred = base_pred + daily_noise
        
        pred = np.clip(pred, 
                       max(0, historical_aqi[-1] - 3 * change_std),
                       min(500, historical_aqi[-1] + 3 * change_std))
        pred = np.clip(pred, 0, 500)
        
        ci_width = change_std * (1 + i * 0.02)
        
        forecasts.append({
            'Date': date,
            'City': city,
            'AQI_Predicted': pred,
            'AQI_Lower': max(0, pred - 1.96 * ci_width),
            'AQI_Upper': min(500, pred + 1.96 * ci_width)
        })
        
        historical_aqi.append(pred)
    
    return pd.DataFrame(forecasts)

print("Forecast function defined: forecast_aqi_with_variability()")

In [ ]:
# Run the initial prototype forecast across 30, 60, and 90-day horizons for all 5 cities.
forecast_horizons = [30, 60, 90]
all_forecasts = {}

print("Generating forecasts...\n")
for days in forecast_horizons:
    forecasts = pd.DataFrame()
    for city in df['City'].unique():
        city_forecast = forecast_aqi_with_variability(
            best_model, df, city, feature_cols, 
            forecast_days=days
        )
        forecasts = pd.concat([forecasts, city_forecast], ignore_index=True)
    all_forecasts[days] = forecasts
    print(f"{days}-day forecast complete")

print("FORECAST SUMMARY")
for days in forecast_horizons:
    print(f"\n{days}-Day Forecast Mean AQI by City:")
    print(all_forecasts[days].groupby('City')['AQI_Predicted'].agg(['mean', 'min', 'max']).round(1))

In [ ]:
# Iteration 1 (Prototype Forecast Plots): Helper functions to visualize 30/60/90-day forecasts per city and across all cities.
def plot_city_forecast(hist_df, forecast_df, city):
    """Plot recent historical AQI alongside the 30-day forecast and confidence interval for a single city."""
    city_hist = hist_df[hist_df['City'] == city].tail(90)
    city_fc = forecast_df[forecast_df['City'] == city]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(city_hist.index, city_hist['AQI'], label='Historical (Last 90d)', alpha=0.7)
    ax.plot(city_fc['Date'], city_fc['AQI_Predicted'], label='30-Day Forecast', color='#e74c3c', linewidth=2)
    if 'AQI_Lower' in city_fc.columns and 'AQI_Upper' in city_fc.columns:
        ax.fill_between(city_fc['Date'], city_fc['AQI_Lower'], city_fc['AQI_Upper'], color='#e74c3c', alpha=0.15, label='95% CI')
    ax.set_title(f'{city} - Historical vs. 30-Day Forecast AQI', fontweight='bold')
    ax.set_ylabel('AQI')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

def plot_all_cities_grid(hist_df, forecasts_by_horizon):
    """Plot a multi-city grid comparing 90-day forecast trajectories across all 5 cities."""
    cities_list = list(hist_df['City'].unique())
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, c in enumerate(cities_list):
        ax = axes[i]
        fc90 = forecasts_by_horizon[90][forecasts_by_horizon[90]['City'] == c]
        ax.plot(fc90['Date'], fc90['AQI_Predicted'], label='90-Day Forecast', color='#3498db')
        ax.set_title(f'{c} (90-Day Horizon)', fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(alpha=0.3)
    axes[-1].axis('off')
    plt.tight_layout()
    return fig

def plot_forecast_comparison_line(hist_df, forecast_30_df):
    """Overlay 30-day forecasted AQI curves for all 5 cities on a single comparison chart."""
    fig, ax = plt.subplots(figsize=(16, 6))
    for c in hist_df['City'].unique():
        sub = forecast_30_df[forecast_30_df['City'] == c]
        ax.plot(sub['Date'], sub['AQI_Predicted'], label=c, linewidth=1.8)
    ax.set_title('30-Day AQI Forecast Comparison Across Cities', fontweight='bold')
    ax.set_ylabel('Predicted AQI')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

for city in df['City'].unique():
    fig = plot_city_forecast(df, all_forecasts[30], city)
    plt.show()

fig = plot_all_cities_grid(df, all_forecasts)
plt.show()

fig = plot_forecast_comparison_line(df, all_forecasts[30])
plt.show()

Final Forecasting System (AQIForecastSystem)

Final consolidated class that unifies recursive multi-day AQI forecasting, seasonal pollutant sampling, dominant pollutant attribution, and India CPCB health categorization.

In [ ]:
# Final Consolidated Class: End-to-end AQI forecasting, pollutant attribution, CPCB categorization, and reporting system.
class AQIForecastSystem:
    """
    Final consolidated forecasting system that combines recursive multi-day AQI prediction,
    seasonal pollutant sampling, dominant pollutant attribution, and CPCB health categorization.
    """
    
    AQI_CATEGORIES = {
        'Good': {'range': (0, 50), 'color': '🟢', 'health': 'Minimal impact'},
        'Satisfactory': {'range': (51, 100), 'color': '🟡', 'health': 'Minor breathing discomfort to sensitive people'},
        'Moderate': {'range': (101, 200), 'color': '🟠', 'health': 'Breathing discomfort to people with lungs, asthma and heart diseases'},
        'Poor': {'range': (201, 300), 'color': '🔴', 'health': 'Breathing discomfort to most people on prolonged exposure'},
        'Very Poor': {'range': (301, 400), 'color': '🟣', 'health': 'Respiratory illness on prolonged exposure'},
        'Severe': {'range': (401, 500), 'color': '⚫', 'health': 'Affects healthy people and seriously impacts those with existing diseases'}
    }
    
    POLLUTANT_WEIGHTS = {
        'PM2.5': 0.30,
        'PM10': 0.25,
        'NO2': 0.15,
        'CO': 0.10,
        'SO2': 0.10,
        'O3': 0.10
    }
    
    CITIES = ['Delhi', 'Mumbai', 'Chennai', 'Kolkata', 'Bangalore']
    
    def __init__(self, model, feature_cols, preprocessor, historical_data):
        """Store the trained model pipeline, feature metadata, and pre-compute city/season pollutant stats."""
        self.model = model
        self.feature_cols = feature_cols
        self.preprocessor = preprocessor
        self.historical_data = historical_data
        
        self._compute_seasonal_stats()
        
        print("AQI Forecast System initialized")
        print(f"   Cities available: {self.CITIES}")
        print("   Max forecast horizon: 90 days")
    
    def _compute_seasonal_stats(self):
        """Pre-compute historical mean, std, and median for each pollutant by city and season."""
        self.seasonal_stats = {}
        
        pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 
                          'Benzene', 'Toluene', 'Xylene']
        
        for city in self.CITIES:
            city_data = self.historical_data[self.historical_data['City'] == city]
            self.seasonal_stats[city] = {}
            
            for season in city_data['Season'].unique():
                season_data = city_data[city_data['Season'] == season]
                self.seasonal_stats[city][season] = {}
                
                for p in pollutant_cols: 
                    if p in city_data.columns:
                        self.seasonal_stats[city][season][p] = {
                            'mean': season_data[p].mean(),
                            'std': season_data[p].std(),
                            'median': season_data[p].median()
                        }
    
    def _get_season(self, city, month):
        """Map a city and calendar month to its corresponding Indian meteorological season."""
        if city in ['Mumbai', 'Bangalore', 'Kolkata']:
            if month in [12, 1, 2]: 
                return 'Winter'
            elif month in [3, 4, 5]: 
                return 'Summer'
            elif month in [6, 7, 8, 9]:
                return 'Monsoon'
            else:
                return 'Post-Monsoon'
        elif city == 'Delhi':
            if month in [12, 1, 2]: 
                return 'Winter'
            elif month in [3, 4, 5, 6]:
                return 'Summer'
            elif month in [7, 8, 9]: 
                return 'Monsoon'
            else:
                return 'Post-Monsoon'
        elif city == 'Chennai':
            if month in [1, 2]: 
                return 'Winter'
            elif month in [3, 4, 5, 6]:
                return 'Summer'
            elif month in [7, 8, 9]: 
                return 'SW-Monsoon'
            else:
                return 'NE-Monsoon'
        else: 
            return 'Summer'
    
    def _get_aqi_category(self, aqi):
        """Classify a numeric AQI value into its CPCB category and health impact description."""
        aqi = float(np.clip(aqi, 0, 500))
        if aqi <= 50:
            cat = 'Good'
        elif aqi <= 100:
            cat = 'Satisfactory'
        elif aqi <= 200:
            cat = 'Moderate'
        elif aqi <= 300:
            cat = 'Poor'
        elif aqi <= 400:
            cat = 'Very Poor'
        else:
            cat = 'Severe'
        return {
            'category': cat,
            'color': self.AQI_CATEGORIES[cat]['color'],
            'health_impact': self.AQI_CATEGORIES[cat]['health']
        }
    
    def _identify_major_pollutant(self, pollutant_values):
        """Identify the primary contributing pollutant using normalized weighted pollutant levels."""
        contributions = {}
        
        for pollutant, weight in self.POLLUTANT_WEIGHTS.items():
            if pollutant in pollutant_values:
                value = pollutant_values[pollutant]
                typical_max = {
                    'PM2.5': 500, 'PM10': 600, 'NO2': 200,
                    'CO': 10, 'SO2': 100, 'O3': 200
                }
                normalized = value / typical_max.get(pollutant, 100)
                contributions[pollutant] = normalized * weight
        
        if contributions:
            major = max(contributions, key=contributions.get)
            return major, contributions[major]
        return 'PM2.5', 0.0
    
    def forecast(self, city, start_date, num_days=30):
        """Generate a multi-day recursive AQI forecast DataFrame for a given city and start date (up to 90 days)."""
        if city not in self.CITIES:
            raise ValueError(f"City must be one of {self.CITIES}")
        
        if num_days > 90:
            print("Max forecast horizon is 90 days. Setting to 90.")
            num_days = 90
        
        if isinstance(start_date, str):
            start_date = pd.to_datetime(start_date)
        
        city_data = self.historical_data[self.historical_data['City'] == city].copy()
        historical_aqi = city_data['AQI'].tolist()
        
        hist_daily_change = city_data['AQI'].diff().dropna()
        change_std = hist_daily_change.std()
        
        dow_pattern = city_data.groupby(city_data.index.dayofweek)['AQI'].mean()
        dow_pattern = dow_pattern / dow_pattern.mean()
        
        forecast_dates = pd.date_range(start=start_date, periods=num_days)
        
        forecasts = []
        pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3',
                          'Benzene', 'Toluene', 'Xylene']
        
        for i, date in enumerate(forecast_dates):
            row = city_data[self.feature_cols].iloc[-1].to_dict()
            
            row['Month'] = date.month
            row['Year'] = date.year
            row['DayOfWeek'] = date.dayofweek
            row['IsWeekend'] = 1 if date.dayofweek in [5, 6] else 0
            
            if 'Month_sin' in self.feature_cols: 
                row['Month_sin'] = np.sin(2 * np.pi * date.month / 12)
            if 'Month_cos' in self.feature_cols:
                row['Month_cos'] = np.cos(2 * np.pi * date.month / 12)
            
            season = self._get_season(city, date.month)
            row['Season'] = season
            row['City'] = city
            
            row['AQI_lag1'] = historical_aqi[-1]
            if 'AQI_roll_mean_7' in self.feature_cols and len(historical_aqi) >= 7:
                row['AQI_roll_mean_7'] = np.mean(historical_aqi[-7:])
            
            pollutant_values = {}
            if city in self.seasonal_stats and season in self.seasonal_stats[city]: 
                for p in pollutant_cols:
                    if p in self.feature_cols and p in self.seasonal_stats[city][season]: 
                        stats = self.seasonal_stats[city][season][p]
                        if pd.notna(stats['mean']) and pd.notna(stats['std']):
                            value = np.random.normal(stats['mean'], stats['std'] * 0.3)
                            value = max(0, value)
                            row[p] = value
                            pollutant_values[p] = value
            
            if 'PM_ratio' in self.feature_cols: 
                row['PM_ratio'] = row.get('PM2.5', 0) / (row.get('PM10', 1) + 1e-8)
            
            row_df = pd.DataFrame([row])[self.feature_cols]
            base_pred = self.model.predict(row_df)[0]
            
            dow = date.dayofweek
            if dow in dow_pattern.index:
                base_pred *= dow_pattern[dow]
            
            noise = np.random.normal(0, change_std * 0.3)
            pred = base_pred + noise
            pred = np.clip(pred, 0, 500)
            
            category_info = self._get_aqi_category(pred)
            major_pollutant, contribution = self._identify_major_pollutant(pollutant_values)
            
            forecasts.append({
                'Date': date.strftime('%Y-%m-%d'),
                'Day': date.strftime('%A'),
                'City': city,
                'AQI': round(pred, 1),
                'Category': category_info['category'],
                'Status': category_info['color'],
                'Major_Pollutant': major_pollutant,
                'Health_Impact': category_info['health_impact']
            })
            
            historical_aqi.append(pred)
        
        return pd.DataFrame(forecasts)
    
    def display_forecast(self, forecast_df):
        """Print a formatted summary and daily table for a forecast DataFrame."""
        print(f"AQI FORECAST: {forecast_df['City'].iloc[0]}")
        print(f"Period: {forecast_df['Date'].iloc[0]} to {forecast_df['Date'].iloc[-1]}")
        
        print("\nFORECAST SUMMARY:")
        print(f"   Average AQI: {forecast_df['AQI'].mean():.1f}")
        print(f"   Min AQI: {forecast_df['AQI'].min():.1f}")
        print(f"   Max AQI: {forecast_df['AQI'].max():.1f}")
        
        print("\nCATEGORY DISTRIBUTION:")
        category_counts = forecast_df['Category'].value_counts()
        for cat, count in category_counts.items():
            pct = count / len(forecast_df) * 100
            color = self.AQI_CATEGORIES[cat]['color']
            print(f"   {color} {cat}: {count} days ({pct:.1f}%)")
        
        print("\nDOMINANT POLLUTANT:")
        major_pollutant_counts = forecast_df['Major_Pollutant'].value_counts()
        for pollutant, count in major_pollutant_counts.head(3).items():
            print(f"   - {pollutant}: {count} days")
        
        print("\nDAILY FORECAST:")
        display_cols = ['Date', 'Day', 'AQI', 'Status', 'Category', 'Major_Pollutant']
        print(forecast_df[display_cols].to_string(index=False))
        
        return forecast_df
    
    def get_forecast_summary(self, forecast_df):
        """Return a structured dictionary summarizing forecast statistics and records for API/deployment use."""
        return {
            'city': forecast_df['City'].iloc[0],
            'start_date': forecast_df['Date'].iloc[0],
            'end_date': forecast_df['Date'].iloc[-1],
            'num_days': len(forecast_df),
            'avg_aqi': round(forecast_df['AQI'].mean(), 1),
            'min_aqi': round(forecast_df['AQI'].min(), 1),
            'max_aqi': round(forecast_df['AQI'].max(), 1),
            'category_distribution': forecast_df['Category'].value_counts().to_dict(),
            'major_pollutant': forecast_df['Major_Pollutant'].mode()[0],
            'forecasts': forecast_df.to_dict('records')
        }

In [ ]:
# Initialize the final AQIForecastSystem instance with the tuned model, preprocessor, and historical data.
print("INITIALIZING AQI FORECAST SYSTEM")

forecast_system = AQIForecastSystem(
    model=best_model,
    feature_cols=feature_cols,
    preprocessor=preprocessor,
    historical_data=df
)

Multi-City Forecast Comparison & Visualization

Helper functions using the final `AQIForecastSystem` class to run single-city forecasts (`predict_aqi`), compare all 5 cities (`compare_cities_forecast`), and plot the 4-panel forecast dashboard (`plot_forecast`).

In [ ]:
# Final Helper Function: Convenience wrapper around AQIForecastSystem.forecast() for single-line predictions.
def predict_aqi(city, start_date, num_days=30, display=True):
    """Validate inputs, run AQIForecastSystem.forecast(), and optionally display the formatted report."""
    valid_cities = ['Delhi', 'Mumbai', 'Chennai', 'Kolkata', 'Bangalore']
    if city not in valid_cities: 
        print(f"Invalid city. Choose from: {valid_cities}")
        return None
    
    if num_days < 1 or num_days > 90:
        print("num_days must be between 1 and 90")
        return None
    
    forecast_df = forecast_system.forecast(city, start_date, num_days)
    
    if display:
        forecast_system.display_forecast(forecast_df)
    
    return forecast_df

print("\nExample 1: 7-day forecast for Bangalore")
short_forecast = predict_aqi('Bangalore', '2025-01-01', 7)

print("\nExample 2: Get forecast data without display")
data = predict_aqi('Chennai', '2025-01-01', 14, display=False)
print(f"   Chennai 14-day average AQI: {data['AQI'].mean():.1f}")
print(f"   Most common category: {data['Category'].mode()[0]}")

In [ ]:
# Final Helper Function: Generate and compare multi-day AQI forecasts across all 5 cities side by side.
def compare_cities_forecast(start_date='2025-01-01', num_days=30):
    """Forecast AQI for all 5 cities over a given window, print a ranked summary table, and plot city comparisons."""
    print(f"CITY COMPARISON: {start_date} to {num_days} days")
    
    all_forecasts = {}
    summary_data = []
    
    for city in forecast_system.CITIES:
        forecast = forecast_system.forecast(city, start_date, num_days)
        all_forecasts[city] = forecast
        
        dominant_cat = forecast['Category'].mode()[0]
        cat_info = forecast_system.AQI_CATEGORIES[dominant_cat]
        
        summary_data.append({
            'City': city,
            'Avg_AQI': round(forecast['AQI'].mean(), 1),
            'Min_AQI': round(forecast['AQI'].min(), 1),
            'Max_AQI': round(forecast['AQI'].max(), 1),
            'Dominant_Category': f"{cat_info['color']} {dominant_cat}",
            'Major_Pollutant': forecast['Major_Pollutant'].mode()[0],
            'Days_Poor+': (forecast['AQI'] > 200).sum()
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values('Avg_AQI')
    
    print("\nCITY COMPARISON SUMMARY:")
    print(summary_df.to_string(index=False))
    
    best_city = summary_df.iloc[0]['City']
    worst_city = summary_df.iloc[-1]['City']
    
    print(f"\nBest Air Quality: {best_city} (Avg AQI: {summary_df.iloc[0]['Avg_AQI']})")
    print(f"Worst Air Quality: {worst_city} (Avg AQI: {summary_df.iloc[-1]['Avg_AQI']})")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1 = axes[0]
    colors = ['#27ae60' if aqi < 100 else '#f1c40f' if aqi < 200 else '#e74c3c' 
              for aqi in summary_df['Avg_AQI']]
    ax1.barh(summary_df['City'], summary_df['Avg_AQI'], color=colors)
    ax1.axvline(x=100, color='orange', linestyle='--', alpha=0.7)
    ax1.axvline(x=200, color='red', linestyle='--', alpha=0.7)
    ax1.set_xlabel('Average AQI')
    ax1.set_title('Average AQI by City', fontsize=14, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    ax2 = axes[1]
    ax2.barh(summary_df['City'], summary_df['Days_Poor+'], color='#e74c3c')
    ax2.set_xlabel('Number of Days')
    ax2.set_title('Days with Poor or Worse AQI', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return all_forecasts, summary_df

all_forecasts, comparison = compare_cities_forecast('2025-01-01', 30)

In [ ]:
# Final Visualization Function: Plot a 4-panel dashboard (daily bar chart, category pie chart, major pollutants, weekday averages) for a forecast DataFrame.
def plot_forecast(forecast_df, figsize=(14, 8)):
    """Render a 4-subplot visual summary of a city's forecasted AQI trajectory, categories, pollutants, and weekly pattern."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    city = forecast_df['City'].iloc[0]
    
    # AQI Over Time
    ax1 = axes[0, 0]
    dates = pd.to_datetime(forecast_df['Date'])
    aqi_values = forecast_df['AQI'].values
    
    colors = []
    color_map = {'Good': '#2ecc71', 'Satisfactory': '#f1c40f', 'Moderate': '#e67e22',
                 'Poor': '#e74c3c', 'Very Poor': '#9b59b6', 'Severe': '#1a1a2e'}
    for cat in forecast_df['Category']: 
        colors.append(color_map.get(cat, '#3498db'))
    
    ax1.bar(dates, aqi_values, color=colors, alpha=0.8, width=0.8)
    ax1.axhline(y=100, color='orange', linestyle='--', alpha=0.7, label='Moderate threshold')
    ax1.axhline(y=200, color='red', linestyle='--', alpha=0.7, label='Poor threshold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('AQI')
    ax1.set_title(f'AQI Forecast: {city}', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Category Distribution
    ax2 = axes[0, 1]
    category_counts = forecast_df['Category'].value_counts()
    colors_pie = [color_map.get(cat, '#3498db') for cat in category_counts.index]
    ax2.pie(category_counts.values, labels=category_counts.index, colors=colors_pie,
            autopct='%1.1f%%', startangle=90)
    ax2.set_title('AQI Category Distribution', fontsize=14, fontweight='bold')
    
    # Major Pollutant
    ax3 = axes[1, 0]
    pollutant_counts = forecast_df['Major_Pollutant'].value_counts()
    ax3.barh(pollutant_counts.index, pollutant_counts.values, color='#3498db')
    ax3.set_xlabel('Number of Days')
    ax3.set_title('Major Pollutant by Days', fontsize=14, fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
    
    # Weekly Pattern
    ax4 = axes[1, 1]
    forecast_df_copy = forecast_df.copy()
    forecast_df_copy['DayOfWeek'] = pd.to_datetime(forecast_df_copy['Date']).dt.day_name()
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekly_aqi = forecast_df_copy.groupby('DayOfWeek')['AQI'].mean().reindex(day_order)
    weekly_aqi.plot(kind='bar', ax=ax4, color='#27ae60', alpha=0.8)
    ax4.set_xlabel('Day of Week')
    ax4.set_ylabel('Average AQI')
    ax4.set_title('Average AQI by Day of Week', fontsize=14, fontweight='bold')
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', alpha=0.3)
    
    plt.suptitle(f'AQI Forecast Analysis: {city}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    return fig

print("FORECAST VISUALIZATION")

delhi_90day = predict_aqi('Delhi', '2025-01-01', 90, display=False)
plot_forecast(delhi_90day)

Saving Final Deployment Package

Serialize the trained model, preprocessing pipeline, feature list, historical data, and category metadata into `aqi_forecast_deployment.joblib`.

In [ ]:
import joblib
from datetime import datetime

In [ ]:
# Final Deployment Artifact Save: Package the trained model, preprocessor, historical data, and category metadata for AQIForecastSystem.
deployment_package = {
    'model': best_model,
    'feature_cols': feature_cols,
    'preprocessor': preprocessor,
    'historical_data': df,
    'cities': forecast_system.CITIES,
    'aqi_categories': forecast_system.AQI_CATEGORIES,
    'metadata': {
        'created_date': pd.Timestamp.now().isoformat(),
        'max_forecast_days': 90,
        'model_type': 'XGBoost (Optuna-tuned)'
    }
}

joblib.dump(deployment_package, 'aqi_forecast_deployment.joblib')
print("\nDeployment package saved to 'aqi_forecast_deployment.joblib'")

print("\nFiles created:")
print("   - aqi_forecast_deployment.joblib - Model and data")
print("   - Use AQIForecastSystem class for predictions")